In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LR_Encoder(nn.Module):
    def __init__(self, in_ch=3, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(nf, nf, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(nf, nf, 3, 1, 1)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
def squeeze2d(x, factor=2):
    B, C, H, W = x.shape
    assert H % factor == 0 and W % factor == 0

    x = x.view(B, C, H // factor, factor, W // factor, factor)
    x = x.permute(0, 1, 3, 5, 2, 4)
    x = x.reshape(B, C * factor * factor, H // factor, W // factor)
    return x


def unsqueeze2d(x, factor=2):
    B, C, H, W = x.shape
    assert C % (factor * factor) == 0

    x = x.view(B, C // (factor * factor), factor, factor, H, W)
    x = x.permute(0, 1, 4, 2, 5, 3)
    x = x.reshape(B, C // (factor * factor), H * factor, W * factor)
    return x

In [2]:
class AffineCoupling(nn.Module):
    def __init__(self, channels, cond_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels//2 + cond_channels, 128, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(128, channels, 3, 1, 1)
        )

    def forward(self, x, cond, reverse=False):
        x1, x2 = torch.chunk(x, 2, dim=1)
        h = torch.cat([x1, cond], dim=1)
        s, t = torch.chunk(self.net(h), 2, dim=1)
        s = torch.tanh(s)

        if not reverse:
            y2 = x2 * torch.exp(s) + t
            log_det = torch.sum(s, dim=[1,2,3])
        else:
            y2 = (x2 - t) * torch.exp(-s)
            log_det = -torch.sum(s, dim=[1,2,3])

        y = torch.cat([x1, y2], dim=1)
        return y, log_det


In [3]:
class FlowBlock(nn.Module):
    def __init__(self, channels, cond_channels):
        super().__init__()
        self.coupling = AffineCoupling(channels, cond_channels)

    def forward(self, x, cond, reverse=False):
        return self.coupling(x, cond, reverse)


In [ ]:
class SRFlow(nn.Module):
    def __init__(self, hr_channels=3, lr_channels=3, nf=64, n_flows=4):
        super().__init__()
        self.encoder = LR_Encoder(lr_channels, nf)

        self.flows = nn.ModuleList([
            FlowBlock(hr_channels, nf)
            for _ in range(n_flows)
        ])

    def forward(self, hr, lr):
        hr = squeeze2d(hr)
        cond = self.encoder(lr)
        log_det_total = 0
        z = hr

        for flow in self.flows:
            z, log_det = flow(z, cond, reverse=False)
            log_det_total += log_det

        return z, log_det_total

    def reverse(self, z, lr):
        z = squeeze2d(z)
        cond = self.encoder(lr)
        x = z

        for flow in reversed(self.flows):
            x, _ = flow(x, cond, reverse=True)

        return x


In [5]:
def nll_loss(z, log_det):
    log_pz = -0.5 * torch.sum(z**2 + torch.log(torch.tensor(2 * 3.1415926)),
                              dim=[1,2,3])
    return -(log_pz + log_det).mean()


In [6]:
model = SRFlow().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for hr, lr in dataloader:
    hr, lr = hr.cuda(), lr.cuda()

    z, log_det = model(hr, lr)
    loss = nll_loss(z, log_det)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


NameError: name 'dataloader' is not defined

In [ ]:
with torch.no_grad():
    lr = lr.cuda()
    z = torch.randn_like(hr) * 0.8   # temperature
    sr = model.reverse(z, lr)
    sr = unsqueeze2d(sr)

# Forward (truyền vào ảnh hr và lr)
# Bước 1: Biến đổi HR  ảnh đầu vào để có số kênh chẵn,tách thành 2 phần x1, x2
# Bước 2: Encode ảnh LR
# Bước 3: h = Concat ảnh embedding LR và x1
# Bước 4: Tách h thành 2 phần s và t
# Bước 5: Chuẩn hóa s và tính toán y2 dựa vào công thức
# Bước 6: Return y=concat(x1, y2), log_det

# Reverse (truyền vào ảnh z~N(0,1) cùng shape với hr và lr)
# Bước 1: Biến đổi z đầu vào để có số kênh chẵn,tách thành 2 phần x1, x2
# Bước 2: Encode ảnh LR
# Bước 3: h = Concat ảnh embedding LR và x1
# Bước 4: Tách h thành 2 phần s và t
# Bước 5: Chuẩn hóa s và tính toán y2 dựa vào công thức
# Bước 6: Return y=concat(x1, y2)